In [ ]:
import pandas as pd
import numpy as np

In [ ]:
gene_list_hsc_e = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/GeneOrdering_mHSC_E.csv")
gene_list_hsc_gm = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/GeneOrdering_mHSC_GM.csv")

tf_list = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/mouse-tfs.csv")
benchmark_chip = pd.read_csv('/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/mHSC-ChIP-seq-network.csv')

In [ ]:
gene_list_hsc_e = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/GeneOrdering_mHSC_E.csv")
gene_list_hsc_gm = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/GeneOrdering_mHSC_GM.csv")

tf_list = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/mouse-tfs.csv")
benchmark_chip = pd.read_csv('/home/gzu5140/Keerthana_b1042/grnInference/real_data/beeline_benchmark/mHSC-ChIP-seq-network.csv')

gene_list_hsc_e[gene_list_hsc_e['VGAMpValue'] < 0.01]
gene_list_hsc_e.columns = ['gene_name', 'VGAMpValue', 'Variance']
gene_list_hsc_e_tf = gene_list_hsc_e[gene_list_hsc_e['gene_name'].isin(list(tf_list['TF']))]
gene_list_hsc_e_tf = gene_list_hsc_e_tf[gene_list_hsc_e_tf['gene_name'].isin(list(benchmark_chip['Gene1']))]
gene_list_hsc_e_gene = gene_list_hsc_e[~gene_list_hsc_e['gene_name'].isin(list(tf_list['TF']))]
gene_list_hsc_gm = gene_list_hsc_gm[gene_list_hsc_gm['VGAMpValue'] < 0.01]
gene_list_hsc_gm.columns = ['gene_name', 'VGAMpValue', 'Variance']
gene_list_hsc_gm_tf = gene_list_hsc_gm[gene_list_hsc_gm['gene_name'].isin(list(tf_list['TF']))]
gene_list_hsc_gm_tf = gene_list_hsc_gm_tf[gene_list_hsc_gm_tf['gene_name'].isin(list(benchmark_chip['Gene1']))]

gene_list_hsc_gm_tf_gene = gene_list_hsc_gm[~gene_list_hsc_gm['gene_name'].isin(list(tf_list['TF']))]

gene_list_hsc_e_gene.shape


In [ ]:
len(list(set(gene_list_hsc_gm_tf_gene['gene_name']).intersection(gene_list_hsc_e_gene['gene_name'])))

In [ ]:
tf_infer_list = gene_list_hsc_e_tf['gene_name'].tolist()
# gene_infer_list = gene_list_hsc_e_gene[:100]['gene_name'].tolist()
full_infer_list = tf_infer_list + top100_common_genes
print(full_infer_list)

In [ ]:
gene_list_hsc_gm = gene_list_hsc_gm[gene_list_hsc_gm['VGAMpValue'] < 0.01]
gene_list_hsc_gm.columns = ['gene_name', 'VGAMpValue', 'Variance']
gene_list_hsc_gm_tf = gene_list_hsc_gm[gene_list_hsc_gm['gene_name'].isin(list(tf_list['TF']))]
gene_list_hsc_gm_tf = gene_list_hsc_gm_tf[gene_list_hsc_gm_tf['gene_name'].isin(list(benchmark_chip['Gene1']))]
gene_list_hsc_gm_tf.shape

In [ ]:
# intersection of gene names
common_genes = set(gene_list_hsc_gm_tf_gene['gene_name']).intersection(
    gene_list_hsc_e_gene['gene_name']
)

# restrict dataframe to intersecting genes
df_common = gene_list_hsc_gm_tf_gene[gene_list_hsc_gm_tf_gene['gene_name'].isin(common_genes)]

# pick top 100 most variable
top100_common = df_common.nlargest(500, 'Variance')

top100_common_genes = top100_common['gene_name'].tolist()

In [ ]:
# ---- 1. Variable genes (significant by VGAM) ----
gene_list_hsc_e.columns  = ['gene_name', 'VGAMpValue', 'Variance']
gene_list_hsc_gm.columns = ['gene_name', 'VGAMpValue', 'Variance']

variable_genes_e  = set(gene_list_hsc_e[gene_list_hsc_e['VGAMpValue']  < 0.01]['gene_name'])
variable_genes_gm = set(gene_list_hsc_gm[gene_list_hsc_gm['VGAMpValue'] < 0.01]['gene_name'])

# Union — gene is variable in at least one lineage
variable_genes = variable_genes_e | variable_genes_gm

# ---- 2. TFs that are also variable ----
tf_set = set(tf_list['TF'])
variable_tfs = tf_set & variable_genes

# ---- 3. Filter ChIP to edges where BOTH Gene1 (TF) and Gene2 (target) are variable ----
chip_filtered = benchmark_chip[
    benchmark_chip['Gene1'].isin(variable_tfs) &      # TF is variable
    benchmark_chip['Gene2'].isin(variable_genes)       # target is variable
]

# ---- 4. Final gene set: all genes appearing in filtered ChIP network ----
final_genes = set(chip_filtered['Gene1']) | set(chip_filtered['Gene2'])

print(f"Variable TFs with ChIP edges:        {len(set(chip_filtered['Gene1']))}")
print(f"Variable targets with ChIP edges:    {len(set(chip_filtered['Gene2']))}")
print(f"Total unique genes (TFs + targets):  {len(final_genes)}")
print(f"ChIP edges retained:                 {len(chip_filtered)}")

# ---- 5. Breakdown by lineage ----
final_genes_e  = final_genes & variable_genes_e
final_genes_gm = final_genes & variable_genes_gm
final_genes_both = variable_genes_e & variable_genes_gm & final_genes

print(f"\nFinal genes variable in E only:      {len(final_genes_e  - variable_genes_gm)}")
print(f"Final genes variable in GM only:     {len(final_genes_gm - variable_genes_e)}")
print(f"Final genes variable in both:        {len(final_genes_both)}")

In [ ]:
# How many unique genes are in the ChIP network total?
print(f"Unique Gene1 (TFs):    {benchmark_chip['Gene1'].nunique()}")
print(f"Unique Gene2 (targets): {benchmark_chip['Gene2'].nunique()}")
print(f"Total edges:            {len(benchmark_chip)}")

# What fraction of possible edges exist?
n_tfs     = benchmark_chip['Gene1'].nunique()
n_targets = benchmark_chip['Gene2'].nunique()
n_edges   = len(benchmark_chip)
print(f"Edge density: {n_edges / (n_tfs * n_targets):.3f}")

# How many targets after filtering to variable genes only?
print(f"\nAfter filtering to variable genes:")
print(f"Unique TFs:     {chip_filtered['Gene1'].nunique()}")
print(f"Unique targets: {chip_filtered['Gene2'].nunique()}")
print(f"Total edges:    {len(chip_filtered)}")

# Preview
print(benchmark_chip.head(10))
print(benchmark_chip.columns.tolist())

In [ ]:
import matplotlib.pyplot as plt

# ---- Count outgoing edges per TF ----
tf_edge_counts = (
    chip_filtered
    .groupby('Gene1')
    .size()
    .reset_index(name='n_targets')
    .sort_values('n_targets', ascending=False)
)

print(tf_edge_counts)

# ---- Plot ----
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(tf_edge_counts['Gene1'], tf_edge_counts['n_targets'], color='0.4')

ax.set_xlabel("TF")
ax.set_ylabel("Number of targets")
ax.set_title("Outgoing ChIP-seq edges per variable TF")

plt.xticks(rotation=90)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numba
import tqdm
import scipy
import os
import sys
import joblib
import scanpy as sc
import os
import itertools
from itertools import product
from collections import defaultdict
from itertools import combinations
from scipy.stats import mannwhitneyu
from pathlib import Path
# from adjustText import adjust_text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
import json

In [ ]:
data_path = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/real_data/"
path_to_plots = f"/home/gzu5140/Keerthana_b1042/grnInference/plots/figure_5/"
os.makedirs(path_to_plots, exist_ok=True)
# path_to_plot_data = Path("/home/gzu5140/Keerthana_b1042/grnInference/analysisData/figure_5/")
# path_to_plot_data.mkdir(exist_ok=True)

In [ ]:
#Load the h5ad file
import subprocess
from pathlib import Path

repo = Path("/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER")

cmd = """
module load git-lfs
git lfs pull --include=real_data/LSK_d2_d4_d6.h5ad
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    cwd=repo,
    capture_output=True,
    text=True,
)

print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
print("Return code:", result.returncode)

# Define the full path to the data file
file_path = f'{data_path}LSK_d2_d4_d6.h5ad'
adata = sc.read_h5ad(file_path)
adata.obs_names_make_unique()
# Plot UMAP
sc.pl.umap(adata, color=['Time point', 'Cell type annotation'])

In [ ]:
t1 = 2
t2 = 4
t3 = 6

#Subset the data into different time points - includes both barcoded and not barcoded cells (hence, all)
adata.obs['cell_id'] = adata.obs.index
adata_t1_all = adata[(adata.obs['Time point'] == t1)].copy()
adata_t2_all = adata[(adata.obs['Time point'] == t2)].copy()
adata_t3_all = adata[(adata.obs['Time point'] == t3)].copy()

In [ ]:
adata.var_names

In [ ]:
#Check that all of these genes are present in the dataset (since sometimes they have alternate names. E.g. Pu1 is names Spi1)
gene_list_check = full_infer_list

all_genes = {g.lower() for g in adata.var_names}
gene_list_check =  [g.lower() for g in gene_list_check]
found = [a for a in gene_list_check if a in all_genes]
not_found = [a for a in gene_list_check if a not in all_genes]
print(f"Found {len(found)} genes out of {len(gene_list_check)}")
if found:
    print("✅ Found gene(s):", found)

print("❌ Not found in dataset", not_found)

In [ ]:
print(found)